# 08 — Counterfactual displacement spectrum (prereg A11)

**BLIND-SAFE up to section 7.** Sections 1–6 score only **random encoders** and the raw-pixel
reference, whose values A4 already makes disclosable. Section 7 scores trained checkpoints and is
gated on amendment **A11** being filed — the module refuses to load a trained checkpoint without
`--amendment`.

## What this measures, and why it is not another probe

Both datasets are complete factorial grids, so for any factor F, values v and v', and any
assignment z of the remaining factors, **both** images `g(z, F=v)` and `g(z, F=v')` exist. Exact
counterfactual pairs with context held fixed are available by construction, so the encoder's
response to changing F alone can be measured with **no probe of any kind**:

    Delta_F(z; v -> v') = W . (h(g(z, F=v')) - h(g(z, F=v)))

whitened by the probe-train covariance. Three statistics:

| statistic | meaning |
|---|---|
| `m_F`   | how far the encoder moves at all. At the null it does not move, so **no probe of any capacity** can recover F — a probe-free destruction certificate. |
| `rho_F` | fraction of displacement energy on one direction. A linear probe reads a factor off one direction, so this **bounds** linear readout quality. |
| `r_F`   | how many directions the factor's effect occupies. |

This is immune to the null saturation that sank the `G`-based headline: it computes no accuracy,
so it has no ceiling.

**Preregistered link (A11 d):** `(1 - rho_F)` predicts `Delta_G`. Confirm iff Spearman
`rho_s >= 0.5` with a bootstrap CI excluding 0, in its **own** Holm family. A failure is reported
as a failure and the destruction certificate stands alone.

## 1. Verify the GPU(s)

In [ ]:
!nvidia-smi

## 2. Clone the repo

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/kaggle/working/probe-capacity-invariance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Install dependencies
Without disturbing Kaggle's preinstalled, CUDA-matched `torch`/`torchvision`.

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())

## 4. Datasets + image cache
`--build-cache` decompresses once into an uncompressed memmap the loaders mmap. Idempotent.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.dsprites --download --build-cache

In [ ]:
# Restore prior outputs + encoder checkpoints so a timed-out session continues.
import shutil
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance"); INPUT = Path("/kaggle/input")
restored = 0
for src in list(INPUT.glob("*/results")) + list(INPUT.glob("*/probe-capacity-invariance/results")):
    for f in src.rglob("*"):
        if f.is_file() and f.suffix in (".npz", ".json", ".jsonl", ".pt"):
            dst = REPO / "results" / f.relative_to(src)
            if not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst); restored += 1
print(f"restored {restored} prior file(s)")

## 5. Preflight — the instrument, and the blind

Checks the A11 module is present, that every §5.4 requirement has a live implementation, and
that the blind guard actually refuses a trained checkpoint without `--amendment`. The last check
is the one that matters: a guard that does not fire is not a guard.

In [ ]:
import inspect
import numpy as np
from src.probes import displacement as D

checks = [
    ("req 1  monotonicity check",      hasattr(D, "monotonicity"), ""),
    ("req 2  fit/eval split",          "n_fit" in inspect.getsource(D.spectrum_stats), ""),
    ("req 2  bootstrap over contexts", "evl_by_ctx" in inspect.signature(D.bootstrap_stats).parameters, ""),
    ("req 3  epsilon_m null",          hasattr(D, "epsilon_m"), ""),
    ("req 4  probe-train whitening",   hasattr(D, "whitener"), ""),
    ("req 6  fixed context seed",      isinstance(getattr(D, "CONTEXT_SEED", None), int), ""),
]
for n, ok, _ in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {n}")
assert all(ok for _, ok, _ in checks), "stale code — git pull, restart kernel"

# the blind guard must FIRE
import argparse
ns = argparse.Namespace(trained_encoders=["x/backbone.pt"], amendment=None, config=None,
                        dataset="shapes3d")
try:
    D.run(ns); raise AssertionError("BLIND GUARD DID NOT FIRE")
except SystemExit as e:
    assert "BLIND GUARD" in str(e), e
    print("\n  PASS  blind guard refuses a trained checkpoint without --amendment")

## 6. Random-encoder + pixel reference (BLIND-SAFE)

This is the null the destruction certificate is measured against, and it needs >= 2 random seeds
or `epsilon_m` is undefined. Roughly `n_contexts x n_values` forward passes per factor — cheap
next to a probe sweep, because nothing is trained.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.displacement \
    --dataset shapes3d --cell reference_shapes3d \
    --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 \
    --n-contexts 512 --device cuda --num-workers 2 --out-root results/displacement

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.displacement \
    --dataset dsprites --cell reference_dsprites \
    --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 \
    --n-contexts 512 --device cuda --num-workers 2 --out-root results/displacement

In [ ]:
import json
from pathlib import Path
import numpy as np

for cell in ("reference_shapes3d", "reference_dsprites"):
    p = Path(f"results/displacement/{cell}.json")
    if not p.exists():
        print(f"{cell}: NOT RUN"); continue
    r = json.loads(p.read_text())
    print(f"\n=== {cell} === contexts={r['n_contexts']} seed={r['context_seed']}")
    for role, tags in r["roles"].items():
        first = next(iter(tags))
        facs = list(tags[first])
        print(f"  {role}:")
        for f in facs:
            ms = [tags[t][f]["m"] for t in tags]
            rhos = [tags[t][f]["rho"] for t in tags]
            mono = [tags[t][f]["monotonicity"]["monotone_fraction"] for t in tags]
            print(f"    {f:14s} m={np.mean(ms):8.4f}  rho={np.mean(rhos):.4f}  "
                  f"mono={np.mean(mono):.2f}  (n_roles={len(tags)})")
    print("  epsilon_m:", {k: round(v, 4) for k, v in r.get("epsilon_m", {}).items()})

## 7. Trained encoders — BLIND-GATED

**Do not run this cell until amendment A11 is filed in `preregistration/prereg.md`.** It is
filed as of 2026-08-13; the guard checks the file rather than trusting this sentence.

Running it produces trained-encoder targeted-factor values. That is exactly what A11 exists to
license, and exactly why the link test's effect-size target (`rho_s >= 0.5`, CI excluding 0) was
fixed in advance.

In [ ]:
from pathlib import Path
AMENDMENT = "A11-2026-08-13"
txt = Path("preregistration/prereg.md").read_text()
assert "### A11 " in txt and "(1 − ρ_F)" in txt.replace("(1 - rho_F)", "(1 − ρ_F)"), (
    "A11 is not in the frozen prereg. File it BEFORE computing rho_F on any trained encoder.")
print("A11 present — trained-encoder displacement is licensed.")
for cell, ds, pat in (("color_strong", "shapes3d", "results/encoders/color_strong_seed*/backbone.pt"),
                      ("control_strong", "shapes3d", "results/encoders/control_strong_seed*/backbone.pt"),
                      ("position_strong", "dsprites", "results/encoders/position_strong_seed*/backbone.pt")):
    print(f"  {cell:16s} {len(sorted(Path().glob(pat))):2d} checkpoint(s)")

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.displacement \
    --dataset shapes3d --cell color_strong --amendment A11-2026-08-13 \
    --trained-encoders results/encoders/color_strong_seed*/backbone.pt \
    --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 \
    --n-contexts 512 --device cuda --num-workers 2 --out-root results/displacement

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.displacement \
    --dataset shapes3d --cell control_strong --amendment A11-2026-08-13 \
    --trained-encoders results/encoders/control_strong_seed*/backbone.pt \
    --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 \
    --n-contexts 512 --device cuda --num-workers 2 --out-root results/displacement

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.displacement \
    --dataset dsprites --cell position_strong --amendment A11-2026-08-13 \
    --trained-encoders results/encoders/position_strong_seed*/backbone.pt \
    --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 \
    --n-contexts 512 --device cuda --num-workers 2 --out-root results/displacement

## 8. The destruction certificate and the A11 (d) link test

`DESTROYED` iff `m_F(random) - m_F(trained) > epsilon_m` — the encoder maps the counterfactual
pair to the same point, so no probe of any capacity can recover F.

The link test needs `Delta_G` from the repaired probe stacks, so run notebook **07 section 9**
first. Its family is the link tests alone (one per dataset x role) and is never pooled with
H1-H4. Target fixed in A11: `rho_s >= 0.5` with a bootstrap CI excluding 0.

In [ ]:
import json
from pathlib import Path
import numpy as np
from scipy.stats import spearmanr

CELLS = ("color_strong", "control_strong", "position_strong")

rows = []
for cell in CELLS:
    dp = Path(f"results/displacement/{cell}.json")
    hp = Path(f"results/probes/{cell}/hypothesis_report.json")
    if not (dp.exists() and hp.exists()):
        print(f"{cell}: missing {'displacement' if not dp.exists() else 'hypothesis_report'}")
        continue
    dj, hj = json.loads(dp.read_text()), json.loads(hp.read_text())
    tr = dj["roles"].get("trained", {})
    rnd = dj["roles"].get("random", {})
    eps_m = dj.get("epsilon_m", {})
    dg = {r["factor"]: r["delta_g"] for r in hj["h1"]["per_factor"]}
    sat = set(hj.get("null_saturation", {}).get("saturated_factors_flip", []))
    diag = set(hj.get("report", {}).get("diagnostic_only_factors", []))
    for f in dg:
        if not tr or f not in next(iter(tr.values()), {}):
            continue
        m_tr = float(np.mean([tr[t][f]["m"] for t in tr]))
        m_rn = float(np.mean([rnd[t][f]["m"] for t in rnd])) if rnd else float("nan")
        rho = float(np.mean([tr[t][f]["rho"] for t in tr]))
        mono = float(np.mean([tr[t][f]["monotonicity"]["monotone_fraction"] for t in tr]))
        e = eps_m.get(f, float("nan"))
        rows.append({"cell": cell, "factor": f, "m_trained": m_tr, "m_random": m_rn,
                     "deficit": m_rn - m_tr, "epsilon_m": e,
                     "destroyed": bool((m_rn - m_tr) > e) if e == e else None,
                     "rho": rho, "monotone_fraction": mono, "delta_g": dg[f],
                     "eligible": f not in sat and f not in diag})

print(f"{'cell':16s}{'factor':14s}{'m(tr)':>9s}{'m(rnd)':>9s}{'deficit':>9s}"
      f"{'eps_m':>8s}{'destr':>7s}{'rho':>7s}{'mono':>7s}{'Delta_G':>9s}{'elig':>6s}")
for r in rows:
    print(f"{r['cell']:16s}{r['factor']:14s}{r['m_trained']:>9.4f}{r['m_random']:>9.4f}"
          f"{r['deficit']:>9.4f}{r['epsilon_m']:>8.4f}{str(r['destroyed']):>7s}"
          f"{r['rho']:>7.3f}{r['monotone_fraction']:>7.2f}{r['delta_g']:>9.4f}"
          f"{str(r['eligible']):>6s}")

elig = [r for r in rows if r["eligible"]]
if len(elig) >= 4:
    x = np.array([1.0 - r["rho"] for r in elig])
    y = np.array([r["delta_g"] for r in elig])
    rs = spearmanr(x, y).statistic
    rng = np.random.default_rng(0)
    boot = [spearmanr(x[i], y[i]).statistic
            for i in (rng.integers(0, len(x), len(x)) for _ in range(2000))]
    lo, hi = np.nanpercentile(boot, [2.5, 97.5])
    confirmed = (rs >= 0.5) and (lo > 0 or hi < 0)
    print(f"\nA11 (d) LINK TEST: spearman rho_s = {rs:+.3f}  CI95 [{lo:+.3f}, {hi:+.3f}]  "
          f"n_readouts = {len(elig)}")
    print("VERDICT:", "CONFIRMED (rho_s >= 0.5 and CI excludes 0)" if confirmed
          else "REFUTED at the preregistered target — report as a refutation. "
               "The destruction certificate stands alone.")
else:
    print(f"\nA11 (d) link test not evaluable: {len(elig)} eligible readout(s), need >= 4.")

## 9. Persist for the next session
Click **Save Version**, then **Add Input -> this output** on the next run.

In [ ]:
import shutil
from pathlib import Path
src = Path("/kaggle/working/probe-capacity-invariance/results"); dst = Path("/kaggle/working/results")
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)
print(f"persisted {sum(1 for _ in dst.rglob('*') if _.is_file())} files -> click 'Save Version'")